In [1]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense

In [4]:
X = np.array([[
    [185, 12],
    [195, 15],
    [200, 20],
    [175, 10],
    [210, 18]
]])
Y = np.array([[
    1,
    1,
    1,
    0,
    0
]])

#0->bad coffe
#1->good coffer

In [5]:
#nomralize features
norm = tf.keras.layers.Normalization(axis=-1)
norm.adapt(X)
Xn= norm(X)
print(np.max(Xn[:, 0]))

-0.6620847


In [8]:
#expand data
Xt = np.tile(Xn, (1000, 1))
Yt = np.tile(Y, (1000, 1))
print(Xt.shape, Yt.shape)

(1, 5000, 2) (1000, 5)


In [10]:
#build model
tf.random.set_seed(1234)
model = Sequential([
    tf.keras.Input(shape=(2,)),
    Dense(3, activation='sigmoid', name='layer1'),
    Dense(1, activation='sigmoid', name='layer2')
])
model.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ layer1 (Dense)                  │ (None, 3)              │             9 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ layer2 (Dense)                  │ (None, 1)              │             4 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 13 (52.00 B)

 Trainable params: 13 (52.00 B)

 Non-trainable params: 0 (0.00 B)

In [11]:
# --- Sanity check parameter counts ---
L1_num_params = 2 * 3 + 3   # W1 + b1
L2_num_params = 3 * 1 + 1   # W2 + b2
print("L1 params = ", L1_num_params, ", L2 params = ", L2_num_params)

# --- Inspect initial (random) weights ---
W1, b1 = model.get_layer("layer1").get_weights()
W2, b2 = model.get_layer("layer2").get_weights()
print(f"W1{W1.shape}:\n", W1, f"\nb1{b1.shape}:", b1)
print(f"W2{W2.shape}:\n", W2, f"\nb2{b2.shape}:", b2)


L1 params =  9 , L2 params =  4
W1(2, 3):
 [[ 0.33068943 -0.00865841 -0.93307585]
 [-0.22211742  0.4200667   0.8777602 ]] 
b1(3,): [0. 0. 0.]
W2(3, 1):
 [[ 0.2734487]
 [ 0.9604806]
 [-0.0097934]] 
b2(1,): [0.]


In [17]:
#compiel and fit mdoel
model.compile(
    loss=tf.keras.losses.BinaryCrossentropy(),
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.01),
)
model.fit(Xt, Yt, epochs=10)


ValueError: Data cardinality is ambiguous. Make sure all arrays contain the same number of samples.'x' sizes: 1
'y' sizes: 1000


In [12]:
# --- Inspect trained weights ---
W1, b1 = model.get_layer("layer1").get_weights()
W2, b2 = model.get_layer("layer2").get_weights()
print("W1:\n", W1, "\nb1:", b1)
print("W2:\n", W2, "\nb2:", b2)

W1:
 [[ 0.33068943 -0.00865841 -0.93307585]
 [-0.22211742  0.4200667   0.8777602 ]] 
b1: [0. 0. 0.]
W2:
 [[ 0.2734487]
 [ 0.9604806]
 [-0.0097934]] 
b2: [0.]


In [15]:
# --- Make predictions ---
X_test = np.array([
    [200, 13.9],  # positive example
    [200, 17]     # negative example
])
X_testn = norm(X_test)
predictions = model.predict(X_test)
print("predictions = \n", predictions)

# --- Convert probabilities to decisions ---
yhat = np.zeros_like(predictions)
for i in range(len(predictions)):
    if predictions[i] >= 0.5:
        yhat[i] = 1
    else:
        yhat[i] = 0
print(f"decisions = \n{yhat}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 266ms/step
predictions = 
 [[0.77177876]
 [0.7737575 ]]
decisions = 
[[1.]
 [1.]]


In [16]:
# Shortcut version of the same thing
yhat = (predictions >= 0.5).astype(int)
print(f"decisions = \n{yhat}")

decisions = 
[[1]
 [1]]
